# 12 - Content Creation Agent

Demonstrates the **orchestrator + worker tools** pattern using `ContentCreationAgent`.

An orchestrator `SimpleAgent` builds content step by step by calling worker sub-agents exposed as MCP tools. The LLM determines how many sections to write and how many times to call each worker.

```
content-creation-agent  (orchestrator)
  ├─ [tool] research_topic   →  research-worker  (SimpleAgent)
  ├─ [tool] create_outline   →  outline-worker   (SimpleAgent)
  └─ [tool] write_section    →  section-writer   (SimpleAgent)
```

The agent class sets up all workers and their MCP server internally.

## Setup

Create an `OllamaClient` and pass it to `ContentCreationAgent`.

In [ ]:
from aimu.models.ollama import OllamaClient
from aimu.agents.examples import ContentCreationAgent

model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
agent = ContentCreationAgent(model_client)

print("Orchestrator model:", model_client.model.value)
print("Worker tools:", [t.name for t in model_client.mcp_client.list_tools()])

## Run

Give the agent a content brief. It will research the topic, build an outline, write each section, then assemble the final piece.

In [ ]:
task = "Write about the benefits of test-driven development as a blog post for software developers"

result = agent.run(task)
print(result)

Inspect the orchestrator's message history to trace the research → outline → write loop.

In [ ]:
agent.messages

## Streaming

`TOOL_CALLING` chunks show which worker was called (and reveal the step-by-step construction); `GENERATING` chunks are the orchestrator assembling the final output.

In [ ]:
from aimu.models import StreamPhase

task = "Write about the benefits of test-driven development as a blog post for software developers"

for chunk in agent.run(task, stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        worker = chunk.content["name"]
        response_preview = chunk.content["response"][:80].replace("\n", " ")
        print(f"\n[tool: {worker}]\n  {response_preview}...\n")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## Clean Up

In [ ]:
del agent, model_client